In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load the California housing dataset (as a proxy for housing prices)
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2, 
                                                    random_state=42)

In [3]:
# Define numerical and categorical features (all features are numerical in this dataset)
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Preprocessing for numerical features: Standardize
numerical_transformer = StandardScaler()

# Preprocessing for categorical features: OneHotEncode (if any)
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [4]:
# Define the models to evaluate
models = {
    'RidgeCV': RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5),
    'LassoCV': LassoCV(alphas=np.logspace(-3, 3, 100), cv=5, max_iter=10000),
    'ElasticNetCV': ElasticNetCV(alphas=np.logspace(-3, 3, 100), l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, max_iter=10000)
}

In [5]:
# Dictionary to store results
results = {}

# Train and evaluate each model
for name, model in models.items():
    # Create pipeline: preprocessor + model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Fit the pipeline
    pipeline.fit(X_train, y_train)

    # Predict on test set
    y_pred = pipeline.predict(X_test)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Store results
    results[name] = {
        'RMSE': rmse,
        'R²': r2,
        'Best Alpha': model.alpha_ if hasattr(model, 'alpha_') else None,
        'Best L1 Ratio': model.l1_ratio_ if hasattr(model, 'l1_ratio_') else None
    }

In [6]:
#Convert results to a DataFrame for comparison
results_df = pd.DataFrame(results).T
print("Model Comparison:")
print(results_df)

# Cross-validation for robustness (optional)
print("\nCross-Validation Scores (R²):")
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    print(f"{name}: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")

Model Comparison:
                  RMSE        R²  Best Alpha  Best L1 Ratio
RidgeCV       0.745581  0.575788       0.001            NaN
LassoCV       0.744642  0.576856       0.001            NaN
ElasticNetCV  0.744697  0.576794       0.001            0.9

Cross-Validation Scores (R²):
RidgeCV: 0.6096 (±0.0099)
LassoCV: 0.6106 (±0.0073)
ElasticNetCV: 0.6104 (±0.0077)


In [7]:
from sklearn.model_selection import GridSearchCV


grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid={'model__alpha': np.logspace(-3, 3, 100)},
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

In [8]:
## give the best alpha from RidgeCV 
best_alpha = models['RidgeCV'].alpha_
print(f"Best alpha from RidgeCV: {best_alpha}")

Best alpha from RidgeCV: 0.001
